# Notebook 14F — Fannie Mae compatibility synthesis
Reconcile the reviewed target and predictor audits, inspect two additional documented fields, and propose an external-validation design. No model fitting, scoring, final feature approval or target changes. Keep all prior notebooks unchanged.

References (check the local download release before approval):
- [Fannie Mae glossary, 29 July 2024](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf), fields 24–37, 40, 44–45, 73, 87.
- [Fannie primary dataset scope](https://capitalmarkets.fanniemae.com/credit-risk-transfer/fannie-mae-single-family-loan-performance-data).
- [Freddie dataset guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf) and [FAQ](https://www.freddiemac.com/fmac-resources/research/pdf/faq.pdf), questions 27–32 and 38.
- [Fannie 2026 credit-score disclosure change](https://capitalmarkets.fanniemae.com/credit-risk-transfer/single-family-credit-risk-transfer/enhance-credit-risk-transfer-disclosures). Do not assume the 2024 layout applies to the local release solely because anchor positions match.

## 1. Verify frozen artifacts and reviewed bundles
Locate the previously saved aggregate outputs. If multiple copies exist, select the exact reviewed ZIP in the overrides below.

In [1]:
from pathlib import Path
import ast, hashlib, json, re, zipfile
from collections import Counter
from functools import lru_cache
from datetime import datetime, timezone
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT/'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
BASE = ROOT/'data/processed/Freddie_Mac/model_baselines_2015_2017'
FEATURE_SOURCE = ROOT/'src/freddie_features.py'
manifest = json.loads((BASE/'model_manifest.json').read_text(encoding='utf-8'))
contract_file = BASE/'freddie_feature_contract.json'
contract = json.loads(contract_file.read_text(encoding='utf-8'))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(contract_file) == manifest['feature_contract_sha256']
assert manifest['horizon_months'] == 24
constants = {t.id: ast.literal_eval(node.value) for node in ast.parse(FEATURE_SOURCE.read_text(encoding='utf-8')).body if isinstance(node,ast.Assign) for t in node.targets if isinstance(t,ast.Name) and t.id in {'NUMERIC_FEATURES','CATEGORICAL_FEATURES'}}
NUM = constants['NUMERIC_FEATURES']; CAT = constants['CATEGORICAL_FEATURES']
assert NUM+CAT == manifest['predictors'] and len(NUM+CAT)==29
assert set(NUM)==set(contract['numeric_medians']) and set(CAT)==set(contract['category_levels'])
assert contract['training_years']==[2015,2016,2017]
print('Verified frozen contract:', len(NUM), 'numeric and', len(CAT), 'categorical predictors')

Verified frozen contract: 15 numeric and 14 categorical predictors


In [2]:
PROCESSED = ROOT/'data/processed/Fannie_Mae'
# Optional explicit paths to the exact reviewed bundles.
OVERRIDES = {}
def locate(name):
    if name in OVERRIDES:
        return Path(OVERRIDES[name])
    matches = sorted(PROCESSED.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Cannot locate {name}; set OVERRIDES.')
    if len({sha(p) for p in matches}) > 1:
        raise ValueError(f'Multiple different bundles: {name}. Set OVERRIDES.')
    return matches[-1]
e_path = locate('notebook_14e_predictor_compatibility_review.zip')
d_path = locate('notebook_14c_stage3de_review.zip')
with zipfile.ZipFile(e_path) as z:
    e_meta = json.loads(z.read('metadata.json'))
    mapping = pd.read_csv(z.open('predictor_mapping.csv'))
    prior_profile = pd.read_csv(z.open('source_field_profile.csv'))
with zipfile.ZipFile(d_path) as z:
    d_meta = json.loads(z.read('stage3de_metadata.json'))
    target = pd.read_csv(z.open('target_summary.csv'))
    cohorts = pd.read_csv(z.open('cohort_target_summary.csv'))
    checks = pd.read_csv(z.open('reconciliation_checks.csv'))
for m in [e_meta,d_meta]:
    assert (m['sample_loans'],m['sample_rows'],m['physical_rows']) == (4749,245552,23563399)
    assert m['file_bytes'] == SOURCE.stat().st_size
    assert not m['models_scored']
assert e_meta['frozen_contract_sha256'] == sha(contract_file)
assert e_meta['feature_source_sha256'] == sha(FEATURE_SOURCE)
assert d_meta['freddie_source_sha256'] == sha(ROOT/'src/freddie_performance.py')
assert mapping.predictor.tolist() == NUM+CAT
assert checks.passed.astype(str).str.lower().eq('true').all()
assert target.set_index('policy').eligible.to_dict()=={'primary':4491,'strict':4488}
assert target.composite.eq(33).all()
print('Reviewed evidence reconciled.'); display(target); display(cohorts)

Reviewed evidence reconciled.


,policy,sample_loans,eligible,ineligible,serious,adverse,both,composite,event_rate
0,primary,4749,4491,258,33,3,3,33,0.007348
1,strict,4749,4488,261,33,3,3,33,0.007353


,policy,origination_quarter,loans,eligible,events,event_rate
0,primary,2017Q1,1,0,0,NaN
1,primary,2017Q2,2,0,0,NaN
2,primary,2017Q3,18,0,0,NaN
3,primary,2017Q4,2068,1838,15,0.008161
4,primary,2018Q1,2655,2653,18,0.006785
5,primary,NaN,5,0,0,NaN
6,strict,2017Q1,1,0,0,NaN
7,strict,2017Q2,2,0,0,NaN
8,strict,2017Q3,18,0,0,NaN
9,strict,2017Q4,2068,1837,15,0.008165


## 2. Audit additional fields
Repeat the identical deterministic loan sample. Positions 73 and 87 are candidates from the 2024 glossary, not certified for the local release. Only the earliest reporting row is used for predictor diagnostics; no future backfill.

In [3]:
SIZE = SOURCE.stat().st_size
WIDTH = 113
assert SIZE == 6928164686, 'Source differs from reviewed file.'
POSITIONS = {'loan_id':2,'report':3,'channel':4,'rate':8,'upb':10,'term':13,'orig':14,'first_payment':15,'ltv':20,'cltv':21,'borrowers':22,'dti':23,'borrower_fico':24,'coborrower_fico':25,'fthb':26,'purpose':27,'property_type':28,'units':29,'occupancy':30,'state':31,'mi':34,'amortization':35,'interest_only':37,'mi_type':73,'high_balance':87}
SAMPLE_MODULUS = 100
MAX_SAMPLE_ROWS = 1_000_000
SOURCE_SIGNATURE = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
@lru_cache(maxsize=100_000)
def selected(loan_id):
    return int.from_bytes(hashlib.blake2b(loan_id,digest_size=8,person=b'Fannie14v1').digest(),'big') % SAMPLE_MODULUS == 0
def scan_source(path, expected_width, progress_every=2_000_000):
    counts = Counter()
    sampled = []
    widths_seen = Counter()
    id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')
    indexes = [p-1 for p in POSITIONS.values()]
    with path.open('rb') as stream:
        for line_number, raw in enumerate(stream, 1):
            parts = raw.rstrip(b'\r\n').split(b'|')
            if line_number == 1:
                parts[0] = parts[0].removeprefix(b'\xef\xbb\xbf')
            counts['physical_rows'] += 1
            widths_seen[len(parts)] += 1
            if len(parts) != expected_width:
                counts['wrong_width'] += 1
                continue
            loan_id = parts[1].strip()
            if not id_pattern.fullmatch(loan_id):
                counts['invalid_loan_id'] += 1
                continue
            counts['valid_id_rows'] += 1
            if selected(loan_id):
                sampled.append(tuple(parts[i].strip().decode('ascii') for i in indexes))
                if len(sampled) > MAX_SAMPLE_ROWS:
                    raise RuntimeError('Sample exceeds memory guard; increase SAMPLE_MODULUS and rerun the entire scan.')
            if progress_every and line_number % progress_every == 0:
                print(f'{line_number:,} rows scanned; {len(sampled):,} sampled rows')
    return pd.DataFrame(sampled, columns=list(POSITIONS)), counts, widths_seen

history, scan_counts, full_widths = scan_source(SOURCE, WIDTH)
assert SOURCE_SIGNATURE == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns), 'Source changed during scan.'
print(dict(scan_counts), dict(full_widths))
assert not scan_counts['wrong_width'] and not scan_counts['invalid_loan_id'], 'Malformed records found; review before continuing.'
assert not history.empty, 'No sampled rows'
print('Sample rows:', len(history), '| Sample distinct loans:', history.loan_id.nunique())
assert scan_counts['physical_rows'] == 23563399
assert len(history) == 245552 and history.loan_id.nunique() == 4749, 'Sample differs from reviewed Stage 2.'

2,000,000 rows scanned; 22,660 sampled rows
4,000,000 rows scanned; 43,404 sampled rows
6,000,000 rows scanned; 63,865 sampled rows
8,000,000 rows scanned; 83,683 sampled rows
10,000,000 rows scanned; 103,966 sampled rows
12,000,000 rows scanned; 123,590 sampled rows
14,000,000 rows scanned; 145,624 sampled rows
16,000,000 rows scanned; 165,802 sampled rows
18,000,000 rows scanned; 186,285 sampled rows
20,000,000 rows scanned; 206,376 sampled rows
22,000,000 rows scanned; 229,575 sampled rows
{'physical_rows': 23563399, 'valid_id_rows': 23563399} {113: 23563399}
Sample rows: 245552 | Sample distinct loans: 4749


In [4]:
assert history.report.str.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}').all()
history['report_idx'] = history.report.str[2:].astype(int)*12 + history.report.str[:2].astype(int)
assert not history.duplicated(['loan_id','report_idx']).any()
first = history.sort_values('report_idx').drop_duplicates('loan_id').copy()
fields = [f for f in POSITIONS if f not in {'loan_id','report'}]
profiles=[]; categories=[]
for field in fields:
    v=first[field]; present=v.ne(''); numeric=pd.to_numeric(v.where(present),errors='coerce')
    distinct=history.groupby('loan_id')[field].nunique(dropna=False)
    nonblank_distinct=history[field].where(history[field].ne('')).groupby(history.loan_id).nunique()
    profiles.append(dict(field=field,physical_position=POSITIONS[field],loans=len(first),blank=int((~present).sum()),nonblank=int(present.sum()),numeric_parseable=int(numeric.notna().sum()),numeric_min=numeric.min(),numeric_max=numeric.max(),loans_with_any_change=int(distinct.gt(1).sum()),loans_with_nonblank_conflict=int(nonblank_distinct.gt(1).sum())))
    if field in ['channel','term','borrowers','fthb','purpose','property_type','units','occupancy','state','amortization','interest_only','mi_type','high_balance']:
        for value,count in v.value_counts().items(): categories.append(dict(field=field,value=value or '[blank]',loans=int(count)))
profile=pd.DataFrame(profiles); category_counts=pd.DataFrame(categories)
display(profile)

,field,physical_position,loans,blank,nonblank,numeric_parseable,numeric_min,numeric_max,loans_with_any_change,loans_with_nonblank_conflict
0,channel,4,4749,0,4749,0,NaN,NaN,1,1
1,rate,8,4749,0,4749,4749,2.75,6.125,0,0
2,upb,10,4749,0,4749,4749,24000.00,1223000.000,0,0
3,term,13,4749,0,4749,4749,96.00,360.000,0,0
4,orig,14,4749,0,4749,4749,12017.00,122017.000,4,4
5,first_payment,15,4749,0,4749,4749,12018.00,122017.000,1,1
6,ltv,20,4749,0,4749,4749,9.00,97.000,0,0
7,cltv,21,4749,0,4749,4749,9.00,105.000,0,0
8,borrowers,22,4749,0,4749,4749,1.00,4.000,0,0
9,dti,23,4749,1,4748,4748,1.00,50.000,1,0


In [5]:
# Check the previous field profiles before interpreting the added positions.
cols=['field','physical_position','loans','blank','nonblank','loans_with_any_change','loans_with_nonblank_conflict']
pd.testing.assert_frame_equal(profile.loc[profile.field.isin(prior_profile.field),cols].reset_index(drop=True),prior_profile[cols].reset_index(drop=True),check_dtype=False)
extra = profile[profile.field.isin(['mi_type','high_balance'])].copy()
mi_cross = first.assign(mi_state=first.mi.map(lambda x: 'blank' if x=='' else 'populated'),mi_type_state=first.mi_type.replace('', '[blank]')).groupby(['mi_state','mi_type_state']).size().reset_index(name='loans')
hb_counts = first.high_balance.replace('', '[blank]').value_counts().rename_axis('value').reset_index(name='loans')
assert mi_cross.loans.sum()==hb_counts.loans.sum()==4749
extra_checks = pd.DataFrame([
    {'field':'mi_type','documented_values':'blank, 1, 2, 3','unexpected_loans':int((~first.mi_type.isin(['','1','2','3'])).sum())},
    {'field':'high_balance','documented_values':'Y, N','unexpected_loans':int((~first.high_balance.isin(['Y','N'])).sum())}
])
display(extra); display(mi_cross); display(hb_counts); display(extra_checks)

,field,physical_position,loans,blank,nonblank,numeric_parseable,numeric_min,numeric_max,loans_with_any_change,loans_with_nonblank_conflict
21,mi_type,73,4749,3388,1361,1361,1.0,2.0,1,1
22,high_balance,87,4749,0,4749,0,NaN,NaN,0,0


,mi_state,mi_type_state,loans
0,blank,[blank],3388
1,populated,1,1216
2,populated,2,145


,value,loans
0,N,4653
1,Y,96


,field,documented_values,unexpected_loans
0,mi_type,"blank, 1, 2, 3",0
1,high_balance,"Y, N",0


## 3. Record provisional predictor decisions
The original model cannot be approved for direct scoring while any predictor is unresolved. A conservative common-feature model is a separate proposed experiment, not a substitute for validation of the original model. Exclusions below are based on semantics, not outcomes or performance.

Fannie 2024 glossary describes how borrower scores are selected, so the minimum of both disclosed scores is a candidate loan-level proxy; the Freddie FAQ does not establish an identical selection algorithm. The 2026 Fannie disclosure change makes release confirmation essential. Fannie field 73 distinguishes MI payer type, with null indicating no MI under the documented layout; test that against field 34 before mapping blanks to zero. Freddie distinguishes zero MI from unavailable MI. Freddie FTHB can be unavailable for non-applicable loans; Fannie N is not necessarily the same missingness convention. Fannie R means refinance, whereas Freddie N explicitly means no cash-out refinance, so retain a proposed common purchase/refinance recode only as a later sensitivity. HARP and IO exclusions are documented for the primary Fannie population, not evidence of equivalence for every release. No program flags are manufactured.

In [6]:
hold = {
'FICO_SCORE':'Confirm release and Freddie loan-level selection; min disclosed scores is a proxy only.',
'FICO_SCORE_MISSING':'Depends on representative score and partial-score policy.',
'MI_PERCENT':'Confirm field 73 applicability and reconcile blank percentage against payer type.',
'HAS_MORTGAGE_INSURANCE':'Depends on MI percentage semantics; no LTV-based imputation.',
'MI_PERCENT_MISSING':'Depends on distinguishing no MI from unavailable.',
'SUPER_CONFORMING_FLAG':'Audit field 87 and reconcile limits and Freddie definition.',
'SPECIAL_ELIGIBILITY_PROGRAM':'No verified common program definition.',
'HARP_INDICATOR':'Primary population restriction supports a candidate N only after release confirmation.',
'INTEREST_ONLY_INDICATOR':'Documented field 37; verify primary population and applicability.',
'FIRST_TIME_HOMEBUYER':'Freddie non-applicable missingness versus Fannie N needs harmonization.',
'LOAN_PURPOSE':'Do not assume Fannie R equals Freddie N; consider common purchase/refinance recode separately.'}
# A conservative starting set, not an approved feature contract.
decisions = mapping[['predictor','source_fields','candidate_status','review_issue']].copy()
decisions['proposed_action'] = decisions.predictor.map(lambda x:'hold_out_pending_semantics' if x in hold else 'common_feature_candidate')
decisions['decision_reason'] = [hold.get(p,r) for p,r in zip(decisions.predictor,decisions.review_issue)]
decisions['final_approved'] = False
common = decisions.loc[decisions.proposed_action.eq('common_feature_candidate'),'predictor'].tolist()
assert len(decisions)==29 and len(common)==18
assert not decisions.final_approved.any()
display(decisions); print('Conservative common-feature candidates:',len(common))

,predictor,source_fields,candidate_status,review_issue,proposed_action,decision_reason,final_approved
0,FICO_SCORE,borrower_fico;coborrower_fico;borrowers,requires_investigation,Verify Freddie loan-level representative score...,hold_out_pending_semantics,Confirm release and Freddie loan-level selecti...,False
1,MI_PERCENT,mi,requires_investigation,Confirm blank versus no insurance versus unava...,hold_out_pending_semantics,Confirm field 73 applicability and reconcile b...,False
2,ORIGINAL_DTI,dti,direct_candidate,"Verify units, valid ranges and missing codes.",common_feature_candidate,"Verify units, valid ranges and missing codes.",False
3,ORIGINAL_LTV,ltv,direct_candidate,"Verify units, valid ranges and missing codes.",common_feature_candidate,"Verify units, valid ranges and missing codes.",False
4,ORIGINAL_INTEREST_RATE,rate,direct_candidate,Original rate in percentage points; verify def...,common_feature_candidate,Original rate in percentage points; verify def...,False
5,LOG_ORIGINAL_UPB,upb,harmonization_candidate,"Use log1p of original UPB, never current UPB.",common_feature_candidate,"Use log1p of original UPB, never current UPB.",False
6,CLTV_MINUS_LTV,cltv;ltv,harmonization_candidate,"Use max(CLTV-LTV,0) after comparable cleaning.",common_feature_candidate,"Use max(CLTV-LTV,0) after comparable cleaning.",False
7,HAS_MORTGAGE_INSURANCE,mi,requires_investigation,Depends on MI blank semantics; Freddie uses MI...,hold_out_pending_semantics,Depends on MI percentage semantics; no LTV-bas...,False
8,HAS_SUBORDINATE_FINANCING,cltv;ltv,harmonization_candidate,Freddie uses CLTV > LTV; verify missingness ru...,common_feature_candidate,Freddie uses CLTV > LTV; verify missingness ru...,False
9,HIGH_LTV_GT_80,ltv,harmonization_candidate,Freddie uses LTV > 80.,common_feature_candidate,Freddie uses LTV > 80.,False


Conservative common-feature candidates: 18


## 4. External evaluation plan and release gate
The 2018Q1 acquisition file is not a full origination cohort. Propose an initial full-file evaluation restricted to actual 2018Q1 originations, explicitly described as a subset of that acquisition file. For broader inference, inventory available subsequent acquisition files and deduplicate loans before forming origination cohorts. Keep the first-payment window and primary/strict sensitivity eligibility rules from 14D; verify event semantics and unknowns before scaling. Use all eligible loans in the selected file(s) if feasible, rather than the 1% diagnostic sample; no event-enriched sampling. Select the population before scoring.

If direct transfer is unsupported, fit a common-feature companion logistic regression and XGBoost using Freddie development data only (2015–2016 fit, 2017 validation, final fit 2015–2017); freeze preprocessing and compare the same companion models on comparable Freddie OOT and Fannie cohorts. Use original model settings or a documented development-only selection protocol. Report ROC-AUC, AP with prevalence, Brier, calibration and uncertainty; never tune on Fannie. Original 29-feature results remain separate. One acquisition-quarter evaluation cannot independently confirm a multi-year drift or retraining finding.

In [7]:
# User may record provenance here when the actual release/download record is available.
RELEASE_EVIDENCE = ''
files = sorted((ROOT/'data/raw/Fannie_Mae').rglob('*.csv'))
inventory = pd.DataFrame([{'relative_path':str(p.relative_to(ROOT)), 'bytes':p.stat().st_size} for p in files])
gates = pd.DataFrame([
('reviewed_target_and_predictor_bundles','passed'),
('release_specific_glossary_and_download_provenance','pending_review'),
('terminal_event_semantic_equivalence_and_RA_difference','pending_review'),
('predictor_semantics_and_category_mappings','pending_review'),
('external_population_and_acquisition_file_coverage','pending_review'),
('final_common_feature_contract','pending_review'),
('model_scoring_authorized','blocked')],columns=['gate','status'])
display(inventory); display(gates)

,relative_path,bytes
0,data\raw\Fannie_Mae\2018Q1\2018Q1.csv,6928164686


,gate,status
0,reviewed_target_and_predictor_bundles,passed
1,release_specific_glossary_and_download_provenance,pending_review
2,terminal_event_semantic_equivalence_and_RA_dif...,pending_review
3,predictor_semantics_and_category_mappings,pending_review
4,external_population_and_acquisition_file_coverage,pending_review
5,final_common_feature_contract,pending_review
6,model_scoring_authorized,blocked


## 5. Save aggregate review bundle
No loan-level records or model artifacts are exported. This notebook does not authorize scoring.

In [8]:
out = PROCESSED/'notebook_14f_compatibility_synthesis'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
for name,table in {'predictor_decisions.csv':decisions,'additional_field_profile.csv':extra,'additional_field_checks.csv':extra_checks,'mi_type_crosscheck.csv':mi_cross,'high_balance_counts.csv':hb_counts,'target_summary.csv':target,'cohort_target_summary.csv':cohorts,'acquisition_file_inventory.csv':inventory,'review_gates.csv':gates}.items():
    table.to_csv(out/name,index=False)
meta = dict(stage='14F',prior_14e_zip_sha256=sha(e_path),prior_14d_zip_sha256=sha(d_path),frozen_contract_sha256=sha(contract_file),sample_loans=len(first),sample_rows=len(history),release_evidence=RELEASE_EVIDENCE,release_verified=False,common_feature_candidates=common,final_approved_predictors=0,direct_transfer_approved=False,companion_design='proposed',models_scored=False)
(out/'metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
review=out/'notebook_14f_compatibility_synthesis_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f!=review: z.write(f,f.name)
print('Review ZIP:',review)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14f_compatibility_synthesis\20261124T160948164721Z\notebook_14f_compatibility_synthesis_review.zip
